# Insurance Application Document Summarization

This notebook demonstrates how to generate natural language summaries from extracted insurance application entities using Azure OpenAI.

## Process Flow:
1. Load extracted entities from JSON
2. Filter and prepare English entity values
3. Format entities for LLM context
4. Generate natural language summary using GPT-4
5. Display and save results

## Requirements:
- Azure OpenAI credentials
- Extracted entity data in JSON format

## Import Required Libraries

Import necessary libraries for data handling, Azure OpenAI integration, and logging.

In [1]:
import os
import json
import logging
from typing import Dict, List, Optional
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

# Configure loggingprint("✓ Libraries imported successfully")

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Configuration

Load Azure OpenAI credentials and configure the client.

In [8]:
# Load environment variables
load_dotenv()

try:
    # Azure OpenAI Configuration
    AZURE_OPENAI_ENDPOINT = os.getenv("GPT_4_1_API_ENDPOINT")
    DEPLOYMENT_NAME = os.getenv("GPT_4_1_API_DEPLOYMENT")
    API_VERSION = os.getenv("GPT_4_1_API_VERSION")
    
    if not all([AZURE_OPENAI_ENDPOINT, DEPLOYMENT_NAME, API_VERSION]):
        raise ValueError("Missing required Azure OpenAI configuration")
    
    # Initialize Azure credential
    credential = DefaultAzureCredential()
    
    # Create token provider function
    def get_azure_ad_token():
        return credential.get_token("https://cognitiveservices.azure.com/.default").token
    
    # Initialize Azure OpenAI client
    client = AzureOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_version=API_VERSION,
        azure_deployment=DEPLOYMENT_NAME,
        azure_ad_token_provider=get_azure_ad_token
    )
    
    print("✓ Azure OpenAI client initialized successfully")
    print(f"  Endpoint: {AZURE_OPENAI_ENDPOINT}")
    print(f"  Deployment: {DEPLOYMENT_NAME}")
    
except Exception as e:
    logger.error(f"Failed to initialize Azure OpenAI client: {e}")
    raise

2026-01-08 16:57:03,941 - INFO - No environment configuration found.
2026-01-08 16:57:03,942 - INFO - ManagedIdentityCredential will use IMDS


✓ Azure OpenAI client initialized successfully
  Endpoint: https://hsbc-demo-resource.openai.azure.com/
  Deployment: gpt-4.1


## Data Loading and Preparation

Load the extracted entities from JSON and prepare them for summarization.

In [9]:
def load_entity_data(file_path: str) -> List[Dict]:
    """
    Load entity data from JSON file.
    
    Args:
        file_path: Path to the JSON file containing entity data
        
    Returns:
        List of dictionaries containing entity data per page
        
    Raises:
        FileNotFoundError: If the file doesn't exist
        json.JSONDecodeError: If the file is not valid JSON
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        logger.info(f"Successfully loaded {len(data)} pages of entity data")
        return data
        
    except FileNotFoundError:
        logger.error(f"File not found: {file_path}")
        raise
    except json.JSONDecodeError as e:
        logger.error(f"Invalid JSON format: {e}")
        raise
    except Exception as e:
        logger.error(f"Error loading entity data: {e}")
        raise

# Load the entity data
data_path = os.path.join("data", "labels.json")
entity_data = load_entity_data(data_path)

print(f"\n✓ Loaded entity data from {len(entity_data)} pages")

2026-01-08 16:57:07,178 - INFO - Successfully loaded 4 pages of entity data



✓ Loaded entity data from 4 pages


In [10]:
def extract_english_entities(pages: List[Dict]) -> Dict[str, str]:
    """
    Extract all English entity values from the page data.
    
    Args:
        pages: List of page dictionaries containing entity data
        
    Returns:
        Dictionary mapping entity names to their English values
    """
    entities = {}
    
    try:
        for page in pages:
            page_num = page.get("page_number", "unknown")
            entity_presence = page.get("entity_presence", {})
            entity_value = page.get("entity_value", {})
            
            # Only include entities that are present
            for entity_name, is_present in entity_presence.items():
                if is_present and entity_name in entity_value:
                    value = entity_value[entity_name]
                    # Only include non-empty English values
                    if value and isinstance(value, str) and value.strip():
                        entities[entity_name] = value.strip()
                        logger.debug(f"Page {page_num}: {entity_name} = {value}")
        
        logger.info(f"Extracted {len(entities)} English entities")
        return entities
        
    except Exception as e:
        logger.error(f"Error extracting entities: {e}")
        raise

# Extract all English entities
extracted_entities = extract_english_entities(entity_data)

# Display extracted entities
print("\n" + "=" * 80)
print("EXTRACTED ENTITIES (English)")
print("=" * 80)
for entity_name, value in extracted_entities.items():
    print(f"{entity_name:<45}: {value}")
print(f"\nTotal entities: {len(extracted_entities)}")

2026-01-08 16:57:08,313 - INFO - Extracted 5 English entities



EXTRACTED ENTITIES (English)
Applicant Name                               : Ms LOK WING CHING
Job Title of Applicant                       : DIRECTOR
Business Registration Number of Employer     : 21893829
Height of Applicant                          : 174 cm
Weight of Applicant                          : 77 kg

Total entities: 5


## Generate Summary

Create a natural language summary of the insurance application using the extracted entities.

In [11]:
def create_summary_prompt(entities: Dict[str, str]) -> str:
    """
    Create a prompt for the LLM to generate a natural language summary.
    
    Args:
        entities: Dictionary of entity names and values
        
    Returns:
        Formatted prompt string
    """
    entity_text = "\n".join([f"- {name}: {value}" for name, value in entities.items()])
    
    prompt = f"""You are an insurance application summarization assistant. Below are the extracted details from an insurance application form.

Your task is to create a clear, professional, and comprehensive natural language summary of the applicant's information.

**Guidelines:**
1. Write in paragraph form, not bullet points
2. Group related information logically (personal details, employment, health, etc.)
3. Use professional language suitable for insurance documentation
4. Include all provided information
5. Do not add any information that is not present in the extracted data
6. If there are measurements, include units

**Extracted Entities:**
{entity_text}

**Summary:**"""
    
    return prompt

def generate_summary(entities: Dict[str, str], model: str = None) -> Optional[str]:
    """
    Generate a natural language summary using Azure OpenAI.
    
    Args:
        entities: Dictionary of entity names and values
        model: Optional model name override
        
    Returns:
        Generated summary text or None if failed
    """
    try:
        if not entities:
            logger.warning("No entities provided for summarization")
            return None
        
        # Create the prompt
        prompt = create_summary_prompt(entities)
        
        logger.info("Generating summary with Azure OpenAI...")
        
        # Call Azure OpenAI
        response = client.chat.completions.create(
            model=model or DEPLOYMENT_NAME,
            messages=[
                {"role": "system", "content": "You are a professional insurance documentation assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,  # Lower temperature for more consistent, factual summaries
            max_tokens=1000
        )
        
        summary = response.choices[0].message.content.strip()
        logger.info(f"Summary generated successfully ({len(summary)} characters)")
        
        return summary
        
    except Exception as e:
        logger.error(f"Error generating summary: {e}")
        return None

print("✓ Summarization functions defined")

✓ Summarization functions defined


In [12]:
# Generate the summary
summary = generate_summary(extracted_entities)

# Display the results
print("\n" + "=" * 80)
print("INSURANCE APPLICATION SUMMARY")
print("=" * 80)

if summary:
    print(f"\n{summary}")
    print("\n" + "=" * 80)
    print(f"\nSummary Statistics:")
    print(f"  Characters: {len(summary)}")
    print(f"  Words: {len(summary.split())}")
    print(f"  Lines: {len(summary.splitlines())}")
else:
    print("\n⚠ Failed to generate summary")

2026-01-08 16:57:12,027 - INFO - Generating summary with Azure OpenAI...
2026-01-08 16:57:12,029 - INFO - Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=REDACTED&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.1 Python/3.13.5 (macOS-26.2-arm64-arm-64bit-Mach-O)'
No body was attached to the request
2026-01-08 16:57:13,627 - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
2026-01-08 16:57:16,079 - INFO - HTTP Request: POST https://hsbc-demo-resource.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"
2026-01-08 16:57:16,087 - INFO - Summary generated successfully (244 characters)



INSURANCE APPLICATION SUMMARY

The applicant, Ms. Lok Wing Ching, holds the position of Director. Her employer is registered under the business registration number 21893829. Ms. Lok's physical measurements are recorded as 174 centimeters in height and 77 kilograms in weight.


Summary Statistics:
  Characters: 244
  Words: 37
  Lines: 1


## Save Results (Optional)

Save the summary and entities to files for later use.

In [14]:
def save_results(summary: str, entities: Dict[str, str], output_dir: str = "output") -> None:
    """
    Save the summary and entities to JSON and text files.
    
    Args:
        summary: The generated summary text
        entities: Dictionary of extracted entities
        output_dir: Directory to save output files
    """
    try:
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Save summary as text file
        summary_path = os.path.join(output_dir, "application_summary.txt")
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write(summary)
        logger.info(f"Summary saved to: {summary_path}")
        
        # Save entities and summary as JSON
        json_path = os.path.join(output_dir, "summarization_results.json")
        results = {
            "summary": summary,
            "extracted_entities": entities,
            "metadata": {
                "total_entities": len(entities),
                "summary_length": len(summary),
                "summary_words": len(summary.split())
            }
        }
        
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        logger.info(f"Results saved to: {json_path}")
        
        print(f"\n✓ Results saved successfully:")
        print(f"  - Summary: {summary_path}")
        print(f"  - JSON: {json_path}")
        
    except Exception as e:
        logger.error(f"Error saving results: {e}")
        raise

print("✓ Save function defined")

✓ Save function defined


In [15]:
# Save the results to files
if summary:
    save_results(summary, extracted_entities)
    print(f"\n📁 Files saved in: {os.path.abspath('output')}")
else:
    print("⚠ No summary to save")

2026-01-08 16:58:51,457 - INFO - Summary saved to: output/application_summary.txt
2026-01-08 16:58:51,458 - INFO - Results saved to: output/summarization_results.json



✓ Results saved successfully:
  - Summary: output/application_summary.txt
  - JSON: output/summarization_results.json

📁 Files saved in: /Users/anishganguli/Documents/Projects/HSBC/PoC/HSBC_IWPB_UW/src/document_summarization/notebooks/output
